# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, exploration, and basic analysis on the FAIR² mass spectrometry dataset using the `mlcroissant` library. All dataset elements are referenced by their unique `@id` field according to the Croissant schema.

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View the dataset metadata object
print("Dataset name:", dataset.metadata.name)
print("Version:", dataset.metadata.version)
print("Description:", dataset.metadata.description)


## 2. Data Overview
Review all available record sets and their respective field `@id`s. All entity definitions use their Croissant `@id` for unambiguous reference.

In [ ]:
# List all RecordSet @id's and a summary of fields (by their @id)
record_sets = list(dataset.metadata.record_sets())

print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    field_ids = [field.id for field in rs.fields]
    field_names = [field.name for field in rs.fields]
    print(f"  Fields [@id]:\n    - " + "\n    - ".join(field_ids))
    print(f"  Fields [names]:\n    - " + "\n    - ".join(field_names))
    print()

## 3. Data Extraction
Load data from each record set as a separate DataFrame. Each DataFrame column corresponds to a field `@id`. Adjust the list of `record_set_ids` as needed based on the overview above.

In [ ]:
# Dynamically get all RecordSet @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded: {record_set_id} with {len(records)} records. Columns: {list(dataframes[record_set_id].columns)}\n")
    else:
        print(f"RecordSet {record_set_id} has no records.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic processing: filtering, normalization, and group statistics. All field/column access is performed using the correct field/column `@id` as shown in the previous sections.

In [ ]:
import numpy as np

# Choose a record set for analysis (use the first loaded record set with data)
if dataframes:
    # Automatically pick the largest non-empty RecordSet
    main_record_set_id = max(dataframes, key=lambda x: len(dataframes[x]))
    df = dataframes[main_record_set_id]
    print(f"\nProceeding with RecordSet: {main_record_set_id}\nColumns: {df.columns.tolist()}")
    
    # Select a numeric field for analysis --
    # Try common mass spectrometry field names; fallback to first numeric column found
    numeric_field_id = None
    for candidate in [
        '@id:ev_prot.coverage_percent', 'coverage_percent',
        '@id:ev_prot.peptide_count', 'peptide_count',
        '@id:ev_prot.mw', 'mw', '@id:ev_prot.pi', 'pi'
    ]:
        if candidate in df.columns:
            numeric_field_id = candidate
            break
    
    # Fallback: choose first float or int column
    if numeric_field_id is None:
        for col in df.select_dtypes(include=[np.number]).columns:
            numeric_field_id = col
            break
    
    if numeric_field_id is None:
        print("Could not find numeric field.\n")
    else:
        print(f"Numeric field for analysis: {numeric_field_id}\n")

        # Threshold-based filtering (e.g., for peptides, MW, coverage, etc.)
        threshold = df[numeric_field_id].dropna().mean() # Use mean as threshold example
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (N={len(filtered_df)})\n")
        print(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field, e.g., by sample or accession
        group_candidates = ['@id:ev_prot.accession', 'accession', '@id:ev_prot.sample', 'sample']
        group_field = None
        for candidate in group_candidates:
            if candidate in df.columns:
                group_field = candidate
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
            print(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset using Matplotlib/Seaborn. Field selection always uses the Croissant schema `@id` as referenced above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field, if available
if 'filtered_df' in locals() and numeric_field_id is not None and len(filtered_df) > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field '{numeric_field_id}' in filtered records")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group field is present, show a boxplot by group
    if group_field is not None:
        plt.figure(figsize=(10, 5))
        order = filtered_df[group_field].value_counts().index[:10]  # up to 10 most frequent groups
        sns.boxplot(
            data=filtered_df,
            x=group_field,
            y=numeric_field_id,
            order=order
        )
        plt.title(f"Boxplot of {numeric_field_id} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to load metadata and records from a Croissant-based FAIR² mass spectrometry dataset using `mlcroissant`. All dataset components—including record sets and fields—are referenced by their `@id` fields, enabling unambiguous and reproducible data processing.

- We explored available record sets and their schema.
- Loaded the main data table, filtered it by a representative numeric field, normalized it, and grouped by sample/accession if applicable.
- Visualized the field distribution and group differences.

For more advanced analysis, refer to the [mlcroissant documentation](https://mlcroissant.ai/) and to the official Croissant schema documentation.